# 06 - K-Reciprocal Re-Ranking
Tune `k1` and `lambda_value` for k-reciprocal reranking on the validation split of the selected hyperparameter-search checkpoint while keeping `k2=6` fixed.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.wandb_utils as wandb_utils
from src.models import load_arcface_model_from_checkpoint
from src.training import compute_map_from_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

device = get_device()
device


device(type='cuda')

## Config


In [2]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

config = {
    "results_csv": Path("../checkpoints/e10_hyperparamter_search/incomplete_random_search_results.csv"),
    "checkpoint_name": None,
    "checkpoint_path": None,
    "output_dir": Path("../checkpoints/e06_k_reciprocal_re_ranking"),
    "num_workers": 0,
    "batch_size": None,
    "k1_values": [10, 15, 20, 25, 30, 35, 40],
    "k2_fixed": 6,
    "lambda_values": [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
    "wandb_run_name": None,
}

config["output_dir"].mkdir(parents=True, exist_ok=True)
config


{'results_csv': PosixPath('../checkpoints/e10_hyperparamter_search/incomplete_random_search_results.csv'),
 'checkpoint_name': None,
 'checkpoint_path': None,
 'output_dir': PosixPath('../checkpoints/e06_k_reciprocal_re_ranking'),
 'num_workers': 0,
 'batch_size': None,
 'k1_values': [10, 15, 20, 25, 30, 35, 40],
 'k2_fixed': 6,
 'lambda_values': [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
 'wandb_run_name': None}

## Load Best Hyperparameter Run


In [3]:
results_df = pd.read_csv(config["results_csv"])

if config["checkpoint_path"] is not None:
    selected_row = None
    checkpoint_path = Path(config["checkpoint_path"])
elif config["checkpoint_name"] is not None:
    selected = results_df.loc[results_df["name"] == config["checkpoint_name"]]
    if selected.empty:
        raise ValueError(f"Checkpoint name not found in results CSV: {config['checkpoint_name']}")
    selected_row = selected.sort_values(["best_map_rerank", "best_map"], ascending=False).iloc[0]
    checkpoint_path = Path(selected_row["checkpoint_path"])
else:
    selected_row = results_df.sort_values(["best_map_rerank", "best_map"], ascending=False).iloc[0]
    checkpoint_path = Path(selected_row["checkpoint_path"])

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

selected_summary = pd.DataFrame([
    {
        "name": checkpoint_path.stem if selected_row is None else selected_row["name"],
        "checkpoint_path": str(checkpoint_path),
        "best_map": None if selected_row is None else selected_row["best_map"],
        "best_map_rerank": None if selected_row is None else selected_row["best_map_rerank"],
        "best_epoch": None if selected_row is None else selected_row["best_epoch"],
    }
])
selected_summary


,name,checkpoint_path,best_map,best_map_rerank,best_epoch
0,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,/sc/projects/sci-zacharatou/mpws2025ez1/wan/ch...,0.909545,0.923758,21


## Build Validation Embeddings Once


In [4]:
model, checkpoint, checkpoint_config = load_arcface_model_from_checkpoint(checkpoint_path, device)
eval_frames = data.build_eval_frames_from_config(checkpoint_config)

input_size = int(checkpoint_config["input_size"])
batch_size = config["batch_size"] or int(checkpoint_config.get("batch_size", 32))
num_workers = int(config["num_workers"])
seed = int(checkpoint_config.get("seed", RANDOM_SEED))
checkpoint_rerank = checkpoint_config.get("rerank", {"enabled": False, "k1": 20, "k2": 6, "lambda_value": 0.3})
total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.backbone.parameters())

val_transform = data.build_transforms(model.backbone, input_size)
val_loader = data.create_eval_loader(
    eval_frames["val_df"],
    eval_frames["data_dir"] / "train" / "train",
    val_transform,
    batch_size=batch_size,
    num_workers=num_workers,
    is_test=False,
    seed=seed,
)

val_embeddings, val_labels = inference.extract_embeddings_with_labels(model, val_loader, device)

pd.DataFrame([
    {
        "checkpoint_name": checkpoint_path.stem,
        "checkpoint_epoch": checkpoint.get("epoch"),
        "data_dir": checkpoint_config.get("data_dir"),
        "val_rows": len(eval_frames["val_df"]),
        "batch_size": batch_size,
        "default_k1": checkpoint_rerank.get("k1", 20),
        "fixed_k2": config["k2_fixed"],
        "default_lambda": checkpoint_rerank.get("lambda_value", 0.3),
    }
])


Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

,checkpoint_name,checkpoint_epoch,data_dir,val_rows,batch_size,default_k1,fixed_k2,default_lambda
0,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,21,../data,379,16,20,6,0.3


## Baseline Metrics


In [5]:
baseline_map = compute_map_from_embeddings(val_embeddings, val_labels, use_rerank=False)
default_rerank_map = compute_map_from_embeddings(
    val_embeddings,
    val_labels,
    use_rerank=checkpoint_rerank.get("enabled", False),
    k1=checkpoint_rerank.get("k1", 20),
    k2=checkpoint_rerank.get("k2", 6),
    lambda_value=checkpoint_rerank.get("lambda_value", 0.3),
)

baseline_df = pd.DataFrame([
    {
        "metric": "baseline_no_rerank",
        "val_map": baseline_map,
    },
    {
        "metric": "checkpoint_default_rerank",
        "val_map": default_rerank_map,
    },
])
baseline_df


,metric,val_map
0,baseline_no_rerank,0.909531
1,checkpoint_default_rerank,0.923744


## Grid Search Over k1 And lambda


In [6]:
search_rows = []
for k1 in config["k1_values"]:
    for lambda_value in config["lambda_values"]:
        val_map_rerank = compute_map_from_embeddings(
            val_embeddings,
            val_labels,
            use_rerank=True,
            k1=k1,
            k2=config["k2_fixed"],
            lambda_value=lambda_value,
        )
        search_rows.append(
            {
                "checkpoint_name": checkpoint_path.stem,
                "k1": k1,
                "k2": config["k2_fixed"],
                "lambda_value": lambda_value,
                "val_map_rerank": val_map_rerank,
                "delta_vs_default_rerank": val_map_rerank - default_rerank_map,
                "delta_vs_no_rerank": val_map_rerank - baseline_map,
            }
        )

search_df = pd.DataFrame(search_rows).sort_values("val_map_rerank", ascending=False).reset_index(drop=True)
search_df.to_csv(config["output_dir"] / "rerank_k1_lambda_results.csv", index=False)
search_df.head(20)


,checkpoint_name,k1,k2,lambda_value,val_map_rerank,delta_vs_default_rerank,delta_vs_no_rerank
0,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.30,0.923744,0.000000,0.014213
1,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.40,0.923425,-0.000319,0.013894
2,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.20,0.923050,-0.000693,0.013519
3,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,15,6,0.20,0.922813,-0.000931,0.013282
4,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.10,0.922718,-0.001026,0.013187
5,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,15,6,0.10,0.922583,-0.001161,0.013052
6,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.50,0.922567,-0.001177,0.013036
7,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,15,6,0.30,0.922401,-0.001343,0.012870
8,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,15,6,0.05,0.922391,-0.001353,0.012860
9,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.05,0.922112,-0.001631,0.012581


## Best Setting


In [7]:
best_row = search_df.iloc[0]
best_summary_df = pd.DataFrame([
    {
        "checkpoint_name": best_row["checkpoint_name"],
        "best_k1": int(best_row["k1"]),
        "fixed_k2": int(best_row["k2"]),
        "best_lambda": float(best_row["lambda_value"]),
        "best_val_map_rerank": float(best_row["val_map_rerank"]),
        "default_val_map_rerank": float(default_rerank_map),
        "baseline_no_rerank": float(baseline_map),
        "gain_vs_default_rerank": float(best_row["delta_vs_default_rerank"]),
        "gain_vs_no_rerank": float(best_row["delta_vs_no_rerank"]),
    }
])
best_summary_df.to_csv(config["output_dir"] / "best_rerank_k1_lambda_result.csv", index=False)

wandb_run_config = {
    **config,
    "checkpoint_name": checkpoint_path.stem,
    "checkpoint_path": checkpoint_path,
    "checkpoint_data_dir": checkpoint_config.get("data_dir"),
    "checkpoint_epoch": checkpoint.get("epoch"),
    "default_rerank_k1": checkpoint_rerank.get("k1", 20),
    "default_rerank_k2": checkpoint_rerank.get("k2", 6),
    "default_rerank_lambda": checkpoint_rerank.get("lambda_value", 0.3),
}
wandb_run_name = config["wandb_run_name"] or f"rerank_{checkpoint_path.stem}_k1_lambda_search"
wandb = wandb_utils.init_wandb(
    wandb_run_config,
    run_name=wandb_run_name,
    param_count=total_params,
    param_count_backbone=backbone_params,
)
wandb.log({"baseline_metrics": wandb.Table(dataframe=baseline_df)})
wandb.log({"rerank_search_results": wandb.Table(dataframe=search_df)})
for row in search_df.itertuples(index=False):
    wandb.log(
        {
            "search/k1": int(row.k1),
            "search/k2": int(row.k2),
            "search/lambda_value": float(row.lambda_value),
            "search/val_map_rerank": float(row.val_map_rerank),
            "search/delta_vs_default_rerank": float(row.delta_vs_default_rerank),
            "search/delta_vs_no_rerank": float(row.delta_vs_no_rerank),
        }
    )
wandb.run.summary["baseline_no_rerank"] = float(baseline_map)
wandb.run.summary["default_val_map_rerank"] = float(default_rerank_map)
wandb.run.summary["best_k1"] = int(best_row["k1"])
wandb.run.summary["fixed_k2"] = int(best_row["k2"])
wandb.run.summary["best_lambda"] = float(best_row["lambda_value"])
wandb.run.summary["best_val_map_rerank"] = float(best_row["val_map_rerank"])
wandb.run.summary["gain_vs_default_rerank"] = float(best_row["delta_vs_default_rerank"])
wandb.run.summary["gain_vs_no_rerank"] = float(best_row["delta_vs_no_rerank"])
wandb.run.summary["results_csv_path"] = str(config["output_dir"] / "rerank_k1_lambda_results.csv")
wandb.run.summary["best_result_csv_path"] = str(config["output_dir"] / "best_rerank_k1_lambda_result.csv")
wandb.finish()
best_summary_df


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /sc/home/maximilian.speer/.netrc


wandb: Currently logged in as: maximilian-speer (maximilian-speer-hasso-plattner-institut) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


search/delta_vs_default_rerank,█████████▇▇▇▇▇▇▇▇▇▇▇▆▆▆▆▅▄▄▄▄▄▄▄▃▃▃▃▂▂▂▁
search/delta_vs_no_rerank,█████████▇▇▇▇▇▇▇▇▇▇▇▆▆▆▅▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁
search/k1,▃▃▃▂▂▂▂▃▃▂▅▁▁▁▁▅▂▅▁▅▅▅▅▆▆▆▆▆▇▆▇▇██▇▇█▇██
search/k2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
search/lambda_value,▄▅▃▃▂▇▄▁▁█▇█▃▅▄▁▄█▅█▇▃▁▄▂▁▅██▇▇▅▇▅▃▂▄▁▃▁
search/val_map_rerank,████████▇▇▇▇▇▇▇▇▇▇▇▇▆▆▆▅▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁
baseline_no_rerank,0.90953
best_k1,20
best_lambda,0.3
best_result_csv_path,../checkpoints/e18_r...
best_val_map_rerank,0.92374


,checkpoint_name,best_k1,fixed_k2,best_lambda,best_val_map_rerank,default_val_map_rerank,baseline_no_rerank,gain_vs_default_rerank,gain_vs_no_rerank
0,eva_unfrozen_rs_08_hlr3e-05_blr3e-05_wd1e-04_d...,20,6,0.3,0.923744,0.923744,0.909531,0.0,0.014213


In [8]:
search_df.pivot(index="k1", columns="lambda_value", values="val_map_rerank")


lambda_value,0.05,0.10,0.20,0.30,0.40,0.50,0.60
k1,,,,,,,
10,0.920092,0.920299,0.920540,0.920432,0.920501,0.918164,0.918344
15,0.922391,0.922583,0.922813,0.922401,0.921233,0.920955,0.919058
20,0.922112,0.922718,0.923050,0.923744,0.923425,0.922567,0.921489
25,0.916068,0.915748,0.916963,0.919088,0.918487,0.918329,0.920657
30,0.908832,0.909184,0.909171,0.910938,0.908773,0.908057,0.908315
35,0.900121,0.902379,0.903629,0.903608,0.906080,0.907633,0.908089
40,0.897827,0.898293,0.899857,0.901042,0.904201,0.905962,0.907715
